# 02 Gesprächsverlauf gezielt mitgeben

LangChain durch Literate Programming lernen

## Konzept vor dem Code

    In Notebook 01 war jede Anfrage einzeln. Jetzt frage ich zuerst nach den vier Notebooks und
    anschliessend: «Was kommt im dritten?» Ohne den Verlauf ist diese zweite Frage mehrdeutig.
    Memory bedeutet hier, dass Python den Verlauf speichert und erneut an das Modell sendet.
    Das Modell selbst erhält dadurch kein dauerhaftes Gedächtnis.

    **Buffer** ist wie ein vollständiges Gesprächsprotokoll: wenig Informationsverlust, wachsender Tokenbedarf.
    **Window** nimmt nur die letzten n Gesprächspaare mit: begrenzter Umfang, ältere Angaben gehen verloren.
    **Summary** ersetzt ältere Teile durch eine Zusammenfassung: kompakter, aber fehleranfällig.
    Meine Auswahl: Brauche ich den genauen Wortlaut, nehme ich Buffer. Ist das Gespräch kurz und nur die
    letzte Aufgabe wichtig, eignet sich Window. Ist es lang und zählt vor allem der Sachstand, kommt
    Summary infrage; wichtige Fakten müssen dann gesondert geprüft werden.

    Für die kleine Lernanwendung wähle ich Buffer und zeige die beiden Alternativen separat.
    Ich verwende `RunnableWithMessageHistory` statt alter Memory-Klassen. Der Speicher liegt nur im
    aktuellen Python-Prozess. Eine `session_id` ordnet Gespräche zu, ist aber keine Zugriffskontrolle.
    Dokumentation: [RunnableWithMessageHistory](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory).

## Lernziele

Nach diesem Notebook kann ich:

- Zustandslosigkeit an zwei Aufrufen zeigen
- Buffer Summary und Window vergleichen
- Gespräche durch session_id voneinander trennen
- Verlauf und Zusammenfassung als unterschiedliche Informationsverluste beurteilen

## Setup

Die folgende Zelle lädt die Einstellungen und zeigt die verwendeten Versionen. Das Modell wird erst in den Experimenten aufgerufen. So lässt sich ein Installationsproblem von einem Problem mit einer Modellantwort unterscheiden. Der Begriff hinter dem Experiment ist bereits oben eingeführt. Alle Notebooks laufen unabhängig in einem frischen Kernel.

In [1]:
import sys, os, json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from modul_assistent.core import ROOT, mode, chat_model

load_dotenv(ROOT / '.env', override=True)
assert (3, 12) <= sys.version_info[:2] < (3, 13), 'Getestete Umgebung: Python 3.12'
for package in ['langchain-core', 'langchain-openai', 'langchain-classic']:
    print(package, version(package))
print('Betriebsart:', mode())

langchain-core 1.7.3
langchain-openai 1.7.3
langchain-classic 1.0.8
Betriebsart: demo


## Experiment

Die History wird in einem Dictionary pro Sitzung gespeichert. Der Platzhalter steht zwischen Systemnachricht und aktueller Frage, damit die Reihenfolge des Gesprächs stimmt.

In [2]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
speicher = {}
def verlauf(session_id):
    return speicher.setdefault(session_id, InMemoryChatMessageHistory())
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Antworte auf Deutsch. Die Notebooks behandeln Prompts, Memory, RAG und Tools in dieser Reihenfolge. Wenn der Bezug fehlt, frage nach.'),
    MessagesPlaceholder('history'), ('human','{frage}')])
llm = chat_model(['Die vier Notebooks behandeln Prompts, Memory, RAG und Tools.',
                  'Das dritte Notebook behandelt RAG.'])
mit_memory = RunnableWithMessageHistory(prompt | llm, verlauf,
    input_messages_key='frage', history_messages_key='history')
config_a = {'configurable': {'session_id': 'sitzung-a'}}
print(mit_memory.invoke({'frage':'Welche vier Notebooks gibt es?'}, config=config_a).content)
print(mit_memory.invoke({'frage':'Was kommt im dritten?'}, config=config_a).content)

Die vier Notebooks behandeln Prompts, Memory, RAG und Tools.


Das dritte Notebook behandelt RAG.


C:\Users\linas\Documents\Codex\2026-10-30\applications-mentioned-by-the-user-appshot\work\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\linas\Documents\Codex\2026-10-30\applications-mentioned-by-the-user-appshot\work\ipykernel_33892\3468316977.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return speicher.setdefault(session_id, InMemoryChatMessageHistory())


Die Speicherung wird unabhängig von der Formulierung der Antwort geprüft. Ein Demo-Modell würde seine vorgegebenen Antworten auch ohne echten Gesprächsbezug ausgeben; deshalb wäre seine passende Antwort allein kein Memory-Nachweis.

In [3]:
assert len(verlauf('sitzung-a').messages) == 4
assert len(verlauf('sitzung-b').messages) == 0
ohne_history = prompt.invoke({'history': [], 'frage':'Was kommt im dritten?'})
mit_history = prompt.invoke({'history':verlauf('sitzung-a').messages, 'frage':'Was kommt im dritten?'})
print('Nachrichten ohne Verlauf:', len(ohne_history.messages))
print('Nachrichten mit Verlauf:', len(mit_history.messages))
print(mit_history.to_string())

Nachrichten ohne Verlauf: 2
Nachrichten mit Verlauf: 6
System: Antworte auf Deutsch. Die Notebooks behandeln Prompts, Memory, RAG und Tools in dieser Reihenfolge. Wenn der Bezug fehlt, frage nach.
Human: Welche vier Notebooks gibt es?
AI: Die vier Notebooks behandeln Prompts, Memory, RAG und Tools.
Human: Was kommt im dritten?
AI: Das dritte Notebook behandelt RAG.
Human: Was kommt im dritten?


Ein Window behält hier ein vollständiges Gesprächspaar. Das schützt die Reihenfolge Human und AI. Die Zahl 1 bedeutet in diesem Beispiel also zwei Nachrichten, nicht einen Token und nicht eine einzelne Nachricht.

In [4]:
def window(messages, paare=1):
    if paare < 1:
        raise ValueError('Mindestens ein Gesprächspaar wählen')
    return messages[-2*paare:]
alles = verlauf('sitzung-a').messages
fenster = window(alles, 1)
print([(m.type,m.content) for m in fenster])
assert len(fenster) == 2

[('human', 'Was kommt im dritten?'), ('ai', 'Das dritte Notebook behandelt RAG.')]


Für Summary fasse ich denselben Verlauf mit einer separaten Chain zusammen und verwende die Zusammenfassung anschliessend wieder als Kontext. Bei langen Gesprächen müsste die Anwendung dies regelmässig auslösen. Der kurze Versuch zeigt den Mechanismus und behauptet keine optimale Zusammenfassungsstrategie.

In [5]:
from langchain_core.output_parsers import StrOutputParser
summary_prompt = ChatPromptTemplate.from_template(
    'Fasse nur die Sachinformationen und offenen Fragen zusammen. Erfinde nichts. Verlauf: {text}')
summary_llm = chat_model(['Die Sitzung betrifft vier Lernnotebooks; das dritte behandelt RAG.'])
summary = (summary_prompt | summary_llm | StrOutputParser()).invoke(
    {'text': '\n'.join(m.content for m in alles)})
print('Zusammenfassung:', summary)
aus_summary = ChatPromptTemplate.from_messages([
    ('system','Gesprächszusammenfassung als Kontext: {summary}'),('human','{frage}')])
print(aus_summary.invoke({'summary':summary,'frage':'Welches Thema hat das dritte Notebook?'}).to_string())
print('Zeichen im Buffer:', sum(len(m.content) for m in alles))
print('Zeichen in Summary:', len(summary), '(Zeichen sind keine Tokens)')

Zusammenfassung: Die Sitzung betrifft vier Lernnotebooks; das dritte behandelt RAG.
System: Gesprächszusammenfassung als Kontext: Die Sitzung betrifft vier Lernnotebooks; das dritte behandelt RAG.
Human: Welches Thema hat das dritte Notebook?
Zeichen im Buffer: 145
Zeichen in Summary: 66 (Zeichen sind keine Tokens)


## Häufige Fehler

Eine fehlende session_id ist ein Konfigurationsfehler. Ich reproduziere ihn mit einem Aufruf ohne Sitzungsangabe. Die Korrektur ist das bereits verwendete config_a.

In [6]:
try:
    mit_memory.invoke({'frage':'Hallo'})
except ValueError as fehler:
    print(type(fehler).__name__, str(fehler)[:350])
else:
    raise AssertionError('Fehlende session_id wurde nicht erkannt')
assert config_a['configurable']['session_id'] == 'sitzung-a'

ValueError Missing keys ['session_id'] in config['configurable'] Expected keys are ['session_id'].When using via .invoke() or .stream(), pass in a config; e.g., chain.invoke({'frage': 'foo'}, {'configurable': {'session_id': '[your-value-here]'}})


Ein stiller Fehler ist noch gefährlicher: Zwei Kennungen bekommen versehentlich dasselbe History-Objekt. Das erste assert unten beweist die unerwünschte Vermischung. Die Funktion verlauf erzeugt dagegen getrennte Objekte.

In [7]:
gemeinsam = InMemoryChatMessageHistory()
def falsch_getrennt(session_id):
    return gemeinsam
assert falsch_getrennt('a') is falsch_getrennt('b')  # beweist den Fehler
assert verlauf('sitzung-a') is not verlauf('sitzung-b')  # prüft die Korrektur
print('Die richtige Implementierung trennt die Sitzungen.')

Die richtige Implementierung trennt die Sitzungen.


## Prompt Logbuch

**Mein eigener erster Prompt ohne KI-Hilfe:**

> Noch von mir einzutragen. Die folgende Endversion ist ein KI-gestützter Vorschlag. Ein nachträglich erfundener Erstprompt würde die geforderte eigene Iteration nicht belegen.

**Vorgeschlagene Endversion:**

```text
Antworte auf Deutsch unter Berücksichtigung des mitgegebenen Gesprächsverlaufs. Fehlt ein eindeutiger Bezug, frage nach statt zu raten.
```

**Begründung des Vorschlags:** Ein Prompt kann keine fehlenden Nachrichten ersetzen. Die entscheidende Verbesserung ist hier der Datenfluss über den History-Platzhalter; Summary benötigt zusätzlich eine klare Regel gegen neue Behauptungen.

**Mein Vergleich nach dem eigenen Versuch:** Hier trage ich Modell, Datum, beide tatsächlichen Antworten und meine Beobachtung ein. Demo-Ausgaben sind kein Modellversuch.

## Zusammenfassung

Der Assistent kann den Verlauf einer Sitzung erneut mitsenden. Buffer, Window und Summary haben unterschiedliche Kosten und Verlustrisiken. Notebook 03 ergänzt die bisher manuell vorgegebenen Modulinformationen durch die Suche in zwei PDF-Unterlagen.

**Selbstkontrolle:** Kann ich die oben genannten Lernziele anhand der ausgeführten Zellen erklären, ohne die Hilfsfunktionen nur abzulesen?

## Übungsaufgabe

Führe zwei Gespräche mit unterschiedlichen session_id-Werten. Vergleiche einen Buffer mit einem Window von einem Paar bei einer Frage, die sich auf die erste Nachricht bezieht. Im Demo-Modus prüfst du die übergebenen Nachrichten; die inhaltliche Reaktion prüfst du im BFH-Modus.

Die folgende Zelle bleibt absichtlich leer und ist für meine eigene Lösung vorgesehen.